# Tutorial 2 — Import and fuse SDShiftGELU in Aidge

This notebook loads the single-component artifact produced by `export_sd_components.ipynb`. It verifies the full path:

```text
PyTorch reference → standard ONNX arithmetic → native Aidge SDShiftGELU
```

Run it with the **`aidge`** environment. Importing `aidge_backend_cpu` is required because it registers the native CPU implementation.

In [1]:
from collections import Counter
from pathlib import Path
import json
import time

import numpy as np
import onnxruntime as ort
import aidge_core
import aidge_onnx
import aidge_backend_cpu  # registers native CPU implementations

EXP = Path.cwd() if Path.cwd().name == 'experiments' else Path.cwd() / 'experiments'
ARTIFACTS = EXP / 'sd_components'
MODEL_PATH = ARTIFACTS / 'models' / 'sdshiftgelu_only.sim.onnx'
DATA_DIR = ARTIFACTS / 'data'
assert MODEL_PATH.exists(), f'Run export_sd_components.ipynb first: {MODEL_PATH}'
WARMUP_RUNS = 10
TIMED_RUNS = 200
print('model:', MODEL_PATH.resolve())

model: /home/tl286648/dev/model/LW-DETR/experiments/sd_components/models/sdshiftgelu_only.sim.onnx


## 1. Load the common input and PyTorch reference

The reference was calculated by the inline PyTorch implementation in the export notebook, so this notebook does not import any LW-DETR Python module.

In [2]:
integer_input = np.load(DATA_DIR / 'sdshiftgelu_integer_input.npy').astype(np.int32)
pytorch_integer_output = np.load(
    DATA_DIR / 'sdshiftgelu_pytorch_integer_output.npy'
).astype(np.int32)
with open(DATA_DIR / 'sdshiftgelu_metadata.json') as file:
    metadata = json.load(file)
input_scale = np.float32(metadata['input_scale'])
output_scale = np.float32(metadata['output_scale'])
pytorch_latency_ms = float(metadata['pytorch_latency_ms'])
float_input = integer_input.astype(np.float32) * input_scale
print('native integer input:', integer_input.shape, integer_input.dtype)
print('PyTorch integer output:', pytorch_integer_output.shape, pytorch_integer_output.dtype)
print(f'PyTorch average latency: {pytorch_latency_ms:.6f} ms')

native integer input: (2, 3, 4) int32
PyTorch integer output: (2, 3, 4) int32
PyTorch average latency: 0.688731 ms


## 2. Run the standard ONNX graph

ONNX Runtime executes the unfused arithmetic graph and provides an independent check of the export.

In [3]:
session = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
onnx_output = session.run(None, {input_name: float_input})[0]
onnx_integer_output = np.rint(onnx_output / output_scale).astype(np.int32)

def average_latency_ms(function, warmup=WARMUP_RUNS, repeats=TIMED_RUNS):
    for _ in range(warmup):
        function()
    start = time.perf_counter()
    for _ in range(repeats):
        function()
    return (time.perf_counter() - start) * 1000.0 / repeats


onnx_latency_ms = average_latency_ms(
    lambda: session.run(None, {input_name: float_input})
)
print('PyTorch codes vs ONNX codes max abs:', int(np.max(
    np.abs(pytorch_integer_output.astype(np.int64) - onnx_integer_output.astype(np.int64))
)))
print(f'ONNX Runtime average latency: {onnx_latency_ms:.6f} ms')

PyTorch codes vs ONNX codes max abs: 0
ONNX Runtime average latency: 0.193528 ms


## 3. Import and fuse in Aidge

Before fusion the model contains many standard arithmetic nodes. The recipe recognizes the complete pattern and replaces it with exactly one native `SDShiftGELU` node.

In [4]:
unfused_model = aidge_onnx.load_onnx(str(MODEL_PATH))
aidge_model = aidge_onnx.load_onnx(str(MODEL_PATH))
before = Counter(node.type() for node in unfused_model.get_nodes())
print('Before fusion:', dict(sorted(before.items())))

aidge_core.recipes.fuse_sd_shift_gelu(aidge_model)
after = Counter(node.type() for node in aidge_model.get_nodes())
print('After fusion: ', dict(sorted(after.items())))
assert after['SDShiftGELU'] == 1, 'Expected exactly one fused SDShiftGELU node'

deployment_counts = Counter(node.type() for node in aidge_model.get_nodes())
print('Integer deployment graph:', dict(sorted(deployment_counts.items())))
assert deployment_counts == {'SDShiftGELU': 1}
aidge_onnx.native_coverage_report(aidge_model)

Before fusion: {'Add': 5, 'Clip': 2, 'Div': 10, 'Floor': 12, 'Max': 2, 'Mul': 8, 'Neg': 5, 'Pow': 2, 'Producer': 20, 'Reciprocal': 1, 'Round': 1, 'Sub': 7}
After fusion:  {'SDShiftGELU': 1}
Integer deployment graph: {'SDShiftGELU': 1}
Native operators: 1 (1 types)
- SDShiftGELU: 1
Generic operators: 0 (0 types)
Native types coverage: 100.0% (1/1)
Native operators coverage: 100.0% (1/1)


(defaultdict(int, {'SDShiftGELU': 1}), defaultdict(int, {}))

## 4. Run the native integer path

The requested table measures all four implementations of the same logical component. PyTorch, ONNX Runtime, and unfused Aidge execute a float carrier reconstructed in memory from the saved integer codes and scale metadata. The export adds no extra boundary `Round`; after fusion, Aidge executes only the native `int32 → int32` kernel. Import, fusion, dtype propagation, backend setup, and scheduler construction are excluded from timing.

In [5]:
def collect_aidge_output(model):
    output_node, output_index = model.get_ordered_outputs()[0]
    return np.array(output_node.get_operator().get_output(output_index))


unfused_model.forward_dtype(dtype=aidge_core.dtype.float32)
unfused_model.set_backend('cpu')
unfused_scheduler = aidge_core.SequentialScheduler(unfused_model)
unfused_input = aidge_core.Tensor(float_input)
run_unfused = lambda: unfused_scheduler.forward(False, [unfused_input])
run_unfused()
aidge_unfused_output = collect_aidge_output(unfused_model)
aidge_unfused_integer_output = np.rint(
    aidge_unfused_output / output_scale
).astype(np.int32)
aidge_unfused_latency_ms = average_latency_ms(run_unfused)

aidge_model.forward_dtype(dtype=aidge_core.dtype.int32)
aidge_model.set_backend('cpu')
scheduler = aidge_core.SequentialScheduler(aidge_model)
aidge_input = aidge_core.Tensor(integer_input)
run_fused = lambda: scheduler.forward(False, [aidge_input])
run_fused()
aidge_fused_output = collect_aidge_output(aidge_model)
assert aidge_fused_output.dtype == np.int32, aidge_fused_output.dtype

aidge_fused_latency_ms = average_latency_ms(run_fused)

comparisons = {
    'PyTorch codes vs ONNX codes': np.max(
        np.abs(pytorch_integer_output.astype(np.int64) - onnx_integer_output.astype(np.int64))
    ),
    'PyTorch codes vs Aidge int32': np.max(
        np.abs(pytorch_integer_output.astype(np.int64) - aidge_fused_output.astype(np.int64))
    ),
    'PyTorch codes vs Aidge unfused': np.max(
        np.abs(pytorch_integer_output.astype(np.int64) - aidge_unfused_integer_output.astype(np.int64))
    ),
}
for name, difference in comparisons.items():
    print(f'{name:27s} max_abs={float(difference):.10g}')

np.testing.assert_array_equal(onnx_integer_output, pytorch_integer_output)
np.testing.assert_array_equal(aidge_unfused_integer_output, pytorch_integer_output)
np.testing.assert_array_equal(aidge_fused_output, pytorch_integer_output)
latencies = {
    'PyTorch fake-quant': pytorch_latency_ms,
    'ONNX Runtime': onnx_latency_ms,
    'Aidge unfused': aidge_unfused_latency_ms,
    'Aidge fused int32': aidge_fused_latency_ms,
}
print(f'\nAverage execution time ({TIMED_RUNS} runs after {WARMUP_RUNS} warm-ups)')
for name, latency in latencies.items():
    print(f'{name:20s} {latency:12.6f} ms')
print('Aidge input dtype: ', integer_input.dtype)
print('Aidge output dtype:', aidge_fused_output.dtype)
print('PASS: native Aidge integer codes match PyTorch and ONNX references exactly.')

PyTorch codes vs ONNX codes max_abs=0
PyTorch codes vs Aidge int32 max_abs=0
PyTorch codes vs Aidge unfused max_abs=0

Average execution time (200 runs after 10 warm-ups)
PyTorch fake-quant       0.688731 ms
ONNX Runtime             0.193528 ms
Aidge unfused            0.950965 ms
Aidge fused int32        0.018078 ms
Aidge input dtype:  int32
Aidge output dtype: int32
PASS: native Aidge integer codes match PyTorch and ONNX references exactly.


## 5. Export the fused native operator to standalone C++

`aidge_export_cpp` uses a separate C++ kernel; it does not call the Aidge CPU backend at runtime. The generated project is therefore standalone.

This example uses the scheduler-level export API because the graph is already quantized. The high-level `export(..., dtype='int8')` API means "run PTQ first", while `dtype='float32'` asks the host adapter to add floating-point boundary casts. Neither behavior is appropriate for an existing native `int32` component.

In [6]:
import re
import shutil
import subprocess

import aidge_export_cpp  # registers the standalone C++ exporters
from aidge_core.export_utils import (
    compile_export, copy_folder, generate_main_cpp, scheduler_export,
)
from aidge_core.hw_model import get_target_by_name
from aidge_core.mem_info import generate_optimized_memory_info

CPP_EXPORT_DIR = ARTIFACTS / 'cpp' / 'sdshiftgelu_int32'
cpp_model = aidge_model.clone()
cpp_model.forward_dtype(dtype=aidge_core.dtype.int32)
cpp_input = aidge_core.Tensor(integer_input)
for input_ref in cpp_model.get_mandatory_inputs():
    input_ref.node.get_operator().set_input(input_ref.index, cpp_input)
cpp_model.forward_dims([cpp_input.dims], allow_data_dependency=True)

assert aidge_export_cpp.has_coverage(cpp_model), (
    'The graph contains an operator without a standalone C++ exporter.'
)
aidge_export_cpp.coverage_report(cpp_model)

cpp_model.set_backend('export_cpp')
cpp_scheduler = aidge_core.SequentialScheduler(cpp_model)
cpp_scheduler.generate_scheduling()

# This folder contains generated files only, so rerunning the cell replaces it.
if CPP_EXPORT_DIR.exists():
    shutil.rmtree(CPP_EXPORT_DIR)
scheduler_export(
    cpp_scheduler,
    CPP_EXPORT_DIR,
    memory_manager=generate_optimized_memory_info,
)
host = get_target_by_name('host')
if host.board_files_path:
    copy_folder(host.board_files_path, CPP_EXPORT_DIR)
generate_main_cpp(
    CPP_EXPORT_DIR, cpp_model, main_template_path=host.main_template_path
)
compile_export(CPP_EXPORT_DIR, build_system=host.build_system)
generated_dir = CPP_EXPORT_DIR
print('Generated and compiled C++ project:', Path(generated_dir).resolve())

Supported operators: 1 (1 types)
- SDShiftGELU: 1
Unsupported operators: 0 (0 types)
Supported types coverage: 100.0% (1/1)


100%|██████████| exported 1/1 nodes [elapsed: 00:00, remaining: 00:00, rate:  4.89/s]

Processing node: _0_SDShiftGELU_0[SDShiftGELU], with backend: export_cpp]


100%|██████████| exported 1/1 nodes [elapsed: 00:00, remaining: 00:00, rate:  4.85/s]


[NOTICE] - Compiling the export (arch=host)
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done
-- Generating done
-- Build files have been written to: /home/tl286648/dev/model/LW-DETR/experiments/sd_components/cpp/sdshiftgelu_int32/build
[ 33%] Building CXX object CMakeFiles/run_export.dir/main.cpp.o
[ 66%] Building CXX object CMakeFiles/run_export.dir/dnn/src/forward.cpp.o
[100%] Linking CXX executable run_export
[100%] Built target run_export
Generated and compiled C++ project: /home/tl286648/dev/model/LW-DETR/experiments/sd_components/cpp/sdshiftgelu_int32


## 6. Run and verify the generated C++ executable

The generated `run_export` program prints the component output codes. This cell parses them and requires exact equality with both PyTorch and native Aidge. It also measures process-level latency; this includes process startup, so it is not directly comparable with the in-process timings above.

In [7]:
cpp_executable = Path(generated_dir) / 'build' / 'run_export'
assert cpp_executable.exists(), cpp_executable

start = time.perf_counter()
completed = subprocess.run(
    [str(cpp_executable)],
    cwd=generated_dir,
    check=True,
    capture_output=True,
    text=True,
)
cpp_process_latency_ms = (time.perf_counter() - start) * 1000.0
print(completed.stdout)

output_text = completed.stdout.split(':', 1)[-1]
cpp_values = np.array(
    [int(value) for value in re.findall(r'(?<![A-Za-z_])-?\d+', output_text)],
    dtype=np.int32,
)
expected_size = pytorch_integer_output.size
assert cpp_values.size == expected_size, (cpp_values.size, expected_size, completed.stdout)
cpp_integer_output = cpp_values.reshape(pytorch_integer_output.shape)

np.testing.assert_array_equal(cpp_integer_output, pytorch_integer_output)
np.testing.assert_array_equal(cpp_integer_output, aidge_fused_output)
print(f'C++ executable process latency: {cpp_process_latency_ms:.6f} ms')
print('C++ output dtype:', cpp_integer_output.dtype)
print('PASS: standalone C++ output matches PyTorch and native Aidge exactly.')

_0_SDShiftGELU_0_output_0:
-160 -185 -231 -270 -338 -345 -399 -400 -396 -342 -240 -120 132 360 756 1080 1632 1938 2484 2964 3420 3960 4440 4800 

C++ executable process latency: 17.231921 ms
C++ output dtype: int32
PASS: standalone C++ output matches PyTorch and native Aidge exactly.


## 7. Optional visualization

Uncomment this cell when `aidge_gui` is installed. The integer deployment graph should contain only one native `SDShiftGELU`. Close the visualizer before rerunning it.

In [8]:
from aidge_gui import visualize
visualize(aidge_model)

[NOTICE] - [aidge-gui] Server started at http://127.0.0.1:8765
[NOTICE] - [aidge-gui] Serving | press Ctrl-C to stop.


(node:705535) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)


[NOTICE] - 
[NOTICE]   [aidge-gui] Stopped.
